# Tome 13, Chapter 11: Additive combinatorics (Python companion)

Exercises first, solutions at the end. We measure doubling constants, test the Plünnecke–Ruzsa inequality and find Petridis's extremal subsets by brute force, watch Freiman's theorem in $\mathbb F_2^n$, run the Balog–Szemerédi–Gowers construction on a set with a hidden interval, and count Elekes's incidences. The computer science application: **randomness extraction from sum-product**, the map $(a,b,c)\mapsto ab+c$ of Barak, Impagliazzo and Wigderson. Requires `numpy`.

In [ ]:
import math, random, itertools
from collections import Counter
import numpy as np
random.seed(2026)

def sumset(A, B): return {a + b for a in A for b in B}
def diffset(A, B): return {a - b for a in A for b in B}
def kfold(A, k):
    S = {0}
    for _ in range(k): S = sumset(S, A)
    return S
def doubling(A): return len(sumset(A, A)) / len(A)
def energy(A):
    r = Counter(a + b for a in A for b in A)
    return sum(v * v for v in r.values())

## Exercises

**P11.1 (★) Doubling constants.** Compute $\frac{|A+A|}{|A|}$ for an arithmetic progression, a proper two-dimensional progression $\{i+1000j\}$, a random subset of half of an interval, a random set of $100$ integers below $10^9$, and a subgroup of $\mathbb F_2^{8}$ (vectors as integers, addition as XOR). Which are bounded as the size grows?

**P11.2 (★★) Plünnecke–Ruzsa and Petridis.** For a few sets $A$ with doubling $K$, compare $|nA-mA|$ with $K^{n+m}|A|$. Then, for a set $B$ (sometimes $B=A$, sometimes the structured part of $A$), find by brute force a subset $X\subseteq A$ minimizing $\frac{|X+B|}{|X|}$ (Lemma 11.6) and check $|X+A+C|\le K'|X+C|$ for random $C$.

**P11.3 (★★) Freiman in $\mathbb F_2^n$.** Build the set $A=V\cup\{e_1,\dots,e_m\}$ of Exercise 11.5 and compare $|\langle A\rangle|$ with the bound $K2^{K^4}|A|$ of Theorem 11.8 and with the covering of Theorem 11.9 by cosets of a subgroup of size at most $|A|$.

**P11.4 (★★) BSG at work.** Let $A=I\cup Q$ with $I=\{1,\dots,n\}$ and $Q$ a set of $n$ powers of $2$ (Exercise 11.4), $n=60$. Compute $E(A)$ and $K=\frac{|A|^3}{E(A)}$, build the graph of popular sums of Theorem 11.12, and run the random choice of Lemma 11.10. What does the set $U=N(y)$ look like, and what is its doubling?

**P11.5 (★) Elekes's incidences.** For $A=\{1,\dots,n\}$ count the incidences between $(A+A)\times(A\cdot A)$ and the lines $y=a(x-b)$ directly, and compare with $n^3$ and with the Szemerédi–Trotter bound.

**P11.6 (★★, computer science) A sum-product condenser.** Work in $\mathbb F_p$ with $p=10007$. A *weak source* is a random variable uniform on an unknown set $A$ with $|A|\approx\sqrt p$ (min-entropy rate $\frac{\log|A|}{\log p}\approx\frac12$). For $A$ an interval, a geometric progression and a random set, compute the exact distribution of $ab+c$ for independent $a,b,c$ uniform on $A$, its support, and its collision entropy rate $\frac{-\log\sum_zP(z)^2}{\log p}$. Compare with $a+b+c$ and $abc$.

## Solutions

### P11.1

In [ ]:
for n in [100, 1000]:
    AP = set(range(0, 3 * n, 3))
    m = int(math.isqrt(n))
    GAP = {i + 1000 * j for i in range(m) for j in range(n // m)}
    half = set(random.sample(range(2 * n), n))
    rnd = set(random.sample(range(10**9), n))
    print(f"n={n}: AP {doubling(AP):.2f}, 2-dim GAP {doubling(GAP):.2f}, half of an interval {doubling(half):.2f}, random {doubling(rnd):.1f}")
H = {v for v in range(256) if v & 0b111 == 0}          # the subgroup of vectors with 3 zero coordinates
print("subgroup of F_2^8 of size", len(H), ": doubling", len({a ^ b for a in H for b in H}) / len(H))

The progression has doubling $\frac{2n-1}n$, just under $2$, the minimum allowed by Lemma 11.1. The two-dimensional progression approaches $4=2^2$ (Exercise 11.2(a)) and a random half of an interval approaches $4$ too, since $A+A$ fills almost all of $[0,4n)$ while $|A|=n$: dense subsets of progressions have bounded doubling. The random set of $100$ numbers below $10^9$ has doubling $50.5=\frac{101}2$, the Sidon value $\frac{|A|+1}2$ of Exercise 11.2(b): with overwhelming probability its sums are all distinct, so its doubling grows linearly with its size. The subgroup has doubling exactly $1$. Bounded doubling is the signature of structure, which is the content of Freiman's theorem.

### P11.2

In [ ]:
examples = {"AP + 3 points": (set(range(10)) | {100, 1000, 10000}, set(range(10))),
            "2-dim GAP 4x3": ({i + 50 * j for i in range(4) for j in range(3)}, None),
            "AP + geometric": (set(range(1, 8)) | {2**k for k in range(5, 11)}, set(range(1, 8)))}
for name, (A, B) in examples.items():
    K = doubling(A)
    line = []
    for n, m in [(2, 0), (3, 0), (2, 1), (2, 2)]:
        S = kfold(A, n); T = kfold(A, m)
        line.append(f"|{n}A-{m}A|={len(diffset(S, T))} <= {K**(n+m)*len(A):.0f}")
    print(f"{name}: |A|={len(A)}, K={K:.2f};", "; ".join(line))
    # Petridis: brute-force minimizer of |X+B|/|X| over non-empty X in A
    B = A if B is None else B
    Al = sorted(A); best = (float('inf'), None)
    for r in range(1, len(Al) + 1):
        for X in itertools.combinations(Al, r):
            q = len(sumset(X, B)) / r
            if q < best[0] - 1e-12: best = (q, set(X))
    Kp, X = best
    ok = all(len(sumset(sumset(X, B), C)) <= Kp * len(sumset(X, C)) + 1e-9
             for C in (set(random.sample(range(-200, 20000), random.randint(1, 6))) for _ in range(300)))
    print(f"   B={'A' if B == A else sorted(B)}: |A+B|/|A| = {len(sumset(A, B))/len(A):.3f}; Petridis X={sorted(X)}, K'={Kp:.3f};"
          f" |X+B+C| <= K'|X+C| for 300 random C: {ok}")

All the Plünnecke–Ruzsa bounds hold, with room to spare: for the progression with three extra points, $|2A-2A|=739$ against $K^4|A|=4165$. The bounds are worst-case: they must hold for every set with doubling $K$, including those of Exercise 11.5, where the iterated sumsets really do grow geometrically.

Petridis's subset is the interesting part. With $B=A$, the minimizer is $A$ itself in these three examples (the minimum ratio is the doubling of $A$). With $B$ the interval inside $A$, the minimizer is the interval $X=B$, with $K'=\frac{|B+B|}{|B|}=1.9$ or $1.857$, much smaller than $\frac{|A+B|}{|A|}$: the extra points of $A$ are bad at absorbing $B$ and the lemma throws them away. This is why Theorem 11.7 gives a bound in terms of $K$ but applies it only to the efficient part $X$. The inequality $|X+B+C|\le K'|X+C|$ held for all $300$ random sets $C$, as Lemma 11.6 guarantees.

### P11.3

In [ ]:
def span(vecs):
    S = {0}
    for v in vecs:
        S |= {s ^ v for s in S}
    return S
for d, m in [(6, 4), (8, 8), (10, 12)]:
    V = set(range(2**d))                                 # the subspace of vectors supported on the first d bits
    E = [1 << (d + i) for i in range(m)]
    A = V | set(E)
    AA = {a ^ b for a in A for b in A}
    K = len(AA) / len(A)
    G = span(A)
    print(f"d={d}, m={m}: |A|={len(A)}, K={K:.2f}, |<A>|/|A| = {len(G)/len(A):.1f}, 2^(K-3) = {2**(K-3):.1f}, "
          f"Ruzsa bound K 2^(K^4) = 2^{math.log2(K) + K**4:.0f}; A lies in {1 + m} cosets of V, |V| = {len(V)} <= |A|")

The doubling grows like $m$, and the subgroup generated grows like $2^m$: the ratio $\frac{|\langle A\rangle|}{|A|}$ is $15$, $248$ and $4049$, above the lower bound $2^{K-3}$ of Exercise 11.5 in each case. Ruzsa's bound $K2^{K^4}$ is astronomically weaker ($2^{531}$ and more); for this question the optimal factor is known to be of order $2^{2K}$ up to polynomial factors in $K$ (Green and Tao, then Chaim Even-Zohar). Marton's conjecture, Theorem 11.9, sidesteps the exponential altogether by asking for a covering instead of a containing subgroup: here $A$ is covered by $m+1$ cosets of $V$, a subgroup of size $2^d\le|A|$, and $m+1\le K$ is polynomial in $K$, as Theorem 11.9 promises ($2K^{12}$ cosets).

### P11.4

In [ ]:
n = 60
I = set(range(1, n + 1)); Q = {2**k for k in range(n + 1, 2 * n + 1)}
A = sorted(I | Q); N = len(A)
E = energy(A); K = N**3 / E
r = Counter(a + b for a in A for b in A)
S = {s for s, v in r.items() if v >= N / (2 * K)}
print(f"|A|={N}, E(A)={E}, K=|A|^3/E={K:.2f}, |A+A|={len(sumset(A, A))}, popular sums |S|={len(S)}")
adj = {a: {b for b in A if a + b in S} for a in A}
edges = sum(len(v) for v in adj.values())
print(f"graph of popular sums: {edges} edges, density {edges / N**2:.3f} (Theorem 11.12 guarantees >= 1/(2K) = {1/(2*K):.3f})")
sizes = Counter()
for y in A:
    U = adj[y]
    if not U: continue
    sizes[(len(U), len(U & I), round(len(diffset(U, U)) / len(U), 2))] += 1
for (u, ui, dd), cnt in sorted(sizes.items(), key=lambda t: -t[1])[:6]:
    print(f"  {cnt:3d} choices of y give |U|={u}, of which {ui} in the interval, |U-U|/|U| = {dd}")

The energy is large, $K=\frac{|A|^3}{E(A)}=10.4$, although $|A+A|=5549$ is almost $\frac{|A|^2}2$: this is Exercise 11.4 in numbers. The popular sums (at least $\frac{|A|}{2K}$ representations) are exactly the sums of two elements of the interval that have many representations, $109$ of them, and the graph of popular sums has density $0.25$, five times the guarantee $\frac1{2K}$.

The random choice of Lemma 11.10 picks a vertex $y$ and keeps its neighbourhood $U=N(y)$. Every non-empty neighbourhood lies inside the interval: $50$ choices of $y$ give the whole interval, the others give sub-intervals of length $55$ to $59$, all with $\frac{|U-U|}{|U|}=1.98$. Elements of $Q$ have no neighbours at all, since a sum involving a power of $2$ has at most two representations. The Balog–Szemerédi–Gowers machine finds the hidden interval in one random step; in general (Lemma 11.11) it needs paths of length three because a single neighbourhood may contain a few badly behaved pairs.

### P11.5

In [ ]:
for n in [5, 10, 20, 40]:
    A = range(1, n + 1)
    P = {(s, q) for s in sumset(A, A) for q in {a * b for a in A for b in A}}
    inc = 0
    for a in A:
        for b in A:
            inc += sum(1 for x in sumset(A, A) if (x, a * (x - b)) in P)
    L = n * n
    st = 4 * len(P)**(2/3) * L**(2/3) + 4 * len(P) + L
    print(f"n={n}: |P|={len(P)}, incidences {inc} (n^3={n**3}), Szemeredi-Trotter bound {st:.0f}")

The incidences exceed $n^3$ slightly: besides the $n$ points $(b+c,ac)$, $c\in A$, a line $y=a(x-b)$ can meet other points of $(A+A)\times(A\cdot A)$, for instance $(b+c,ac)$ with $c\notin A$ but $b+c\in A+A$ and $ac\in A\cdot A$. The Szemerédi–Trotter bound is about ten times larger than the count. Elekes's argument uses only the lower bound $n^3$; for $A=\{1,\dots,n\}$ the point set has $|A+A||A\cdot A|$ elements, $40843$ for $n=40$, far above the guaranteed $\frac{n^{5/2}}{64}=158$. Intervals are not where the sum-product problem is hard: the difficult sets are those, like the smooth numbers of Figure 11.2, that are good at both operations at once.

### P11.6

In [ ]:
p = 10007
def dist(vals, weights=None):
    c = np.bincount(np.asarray(vals) % p, minlength=p).astype(float)
    return c / c.sum()
def report(name, A):
    A = np.array(sorted(A)); m = len(A)
    a, b, c = np.meshgrid(A, A, A, indexing="ij")
    rate = lambda P: -math.log(np.sum(P**2)) / math.log(p)
    out = {"ab+c": dist((a * b + c).ravel()), "a+b+c": dist((a + b + c).ravel()), "abc": dist((a * b % p * c).ravel())}
    print(f"{name:22s} |A|={m}, input rate {math.log(m)/math.log(p):.3f}; " +
          ", ".join(f"{k}: support {np.count_nonzero(P)}, rate {rate(P):.3f}" for k, P in out.items()))
m = 100
report("interval {1..100}", range(1, m + 1))
g = 5                                     # 5 is a primitive root mod 10007
report("geometric {5^i}", [pow(g, i, p) for i in range(m)])
report("random set", random.sample(range(1, p), m))

Each weak source has min-entropy rate $\frac12$: it is uniform on $100$ of the $10007$ elements. The output of $(a,b,c)\mapsto a+b+c$ stays concentrated when the source is an interval (support $298$, rate $0.565$), because intervals are closed under addition up to a constant factor; likewise $abc$ stays concentrated on a geometric progression (support $298$, rate $0.565$). But $ab+c$ spreads both: rate $0.931$ for the interval (support all of $\mathbb F_p$) and $0.930$ for the geometric progression. No set can be good at both operations, so a function that uses both cannot be fooled by a structured source; for a random set every combination already spreads (rates near $1$).

This is the idea of Barak, Impagliazzo and Wigderson: the sum-product theorem over $\mathbb F_p$ (Bourgain, Katz and Tao) guarantees that the entropy rate of $ab+c$ is a fixed amount larger than that of the inputs, as long as it is not already close to $1$. Applying the map recursively to independent sources raises the rate until the output is close to uniform, giving a *condenser*, then an *extractor*, that uses a constant number of independent weak sources and no truly random seed.